# HH-HM Correlation Analysis

This notebook reads the exported question-level HH/HM SBERT points and correlation tables for the all-model and matched 7/8B analyses.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

OUT = Path("figures/hh_hm_correlation_scatter")
GROUP_LINE_COLORS = {
    "VLM": "#1f77b4",
    "VLM backbone decoder": "#2ca02c",
    "standalone LLM": "#ff7f0e",
    "standalone LLM (think)": "#d62728",
}
VARIANT_ORDER = ["C", "B", "A"]

def load_scope(scope_label="all_models", q_label="q113_yesno"):
    points = pd.read_csv(OUT / f"hh_hm_sbert_correlation_points_{scope_label}_{q_label}.csv")
    corr = pd.read_csv(OUT / f"hh_hm_sbert_correlation_stats_{scope_label}_{q_label}.csv")
    return points, corr

points_all, corr_all = load_scope("all_models")
points_7b, corr_7b = load_scope("7b")

In [ ]:
display(corr_all.query("scope_type == 'overall' or scope_type == 'group'").sort_values(["variant", "scope_type", "scope"]))
display(corr_all.query("scope_type == 'family'").sort_values(["variant", "scope"]))

In [ ]:
def plot_group_lines(points, title):
    fig, axes = plt.subplots(1, 3, figsize=(17, 5.5), sharex=True, sharey=True)
    lim_min = max(0.0, min(points["hm_sbert"].min(), points["hh_sbert"].min()) - 0.02)
    lim_max = min(1.0, max(points["hm_sbert"].max(), points["hh_sbert"].max()) + 0.02)
    for ax, variant in zip(axes, VARIANT_ORDER):
        sub = points[points["variant"] == variant].copy()
        ax.scatter(sub["hm_sbert"], sub["hh_sbert"], s=12, alpha=0.18, color="#777777")
        for group, gsub in sub.groupby("model_group"):
            if len(gsub) < 3:
                continue
            x = gsub["hm_sbert"].to_numpy()
            y = gsub["hh_sbert"].to_numpy()
            slope, intercept = np.polyfit(x, y, 1)
            xs = np.linspace(float(x.min()), float(x.max()), 100)
            ys = slope * xs + intercept
            ax.plot(xs, ys, lw=2.4, color=GROUP_LINE_COLORS.get(group, "#444444"), label=group)
        ax.plot([lim_min, lim_max], [lim_min, lim_max], ls="--", lw=1.0, color="#bdbdbd")
        ax.set_title(f"Variant {variant}")
        ax.set_xlim(lim_min, lim_max)
        ax.set_ylim(lim_min, lim_max)
        ax.grid(alpha=0.15)
    axes[0].set_ylabel("HH SBERT")
    for ax in axes:
        ax.set_xlabel("HM SBERT")
    handles, labels = axes[0].get_legend_handles_labels()
    uniq = dict(zip(labels, handles))
    fig.legend(uniq.values(), uniq.keys(), loc="lower center", ncol=4, frameon=True)
    fig.suptitle(title)
    plt.tight_layout(rect=[0, 0.08, 1, 0.93])
    plt.show()

plot_group_lines(points_all, "All-model HH-HM correlation with group trend lines")
plot_group_lines(points_7b, "Matched 7/8B HH-HM correlation with group trend lines")